# 02 — Baseline Model: Calendar-Only SARIMAX

This is the "weather-blind" baseline: load history plus calendar features
(hour of day, day of week, time of year, holiday flag) — no temperature.
This is the model we expect to underperform on extreme-temperature days,
which is what the next notebook measures.

**A note on the model choice.** The brief says "SARIMA or Prophet." A classic
SARIMA with a seasonal order at 24-hour periodicity is not practical here —
fitting a seasonal state-space model at m=24 over several years of hourly
data (tens of thousands of points) does not scale, the fit can take a very
long time or fail to converge. Instead this uses SARIMAX with a small
ARIMA(2,1,2) error structure plus Fourier terms (hour-of-day, day-of-year)
and day-of-week/holiday dummies as exogenous regressors — same calendar-only
information the brief asks for, just parameterized so it actually fits in a
reasonable time. Worth having as a ready answer if asked about this choice.

In [ ]:
# lets src/ be imported when running this notebook from the notebooks/ folder
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src import baseline_model, utils

actual, predicted = baseline_model.run_baseline()

## Predicted vs actual — first two weeks of the test year

Plotting the full test year is unreadable, zooming into a short window instead.

In [ ]:
window = slice(0, 24 * 14)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actual.index[window], actual.iloc[window], label="actual", linewidth=1)
ax.plot(predicted.index[window], predicted.iloc[window], label="predicted", linewidth=1)
ax.legend()
ax.set_title("Baseline model — first two weeks of test year")
ax.set_ylabel("MW")
plt.show()

## Residuals over the full test year

Looking for whether errors cluster around particular seasons or dates.

In [ ]:
residual = actual - predicted
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(residual.index, residual, linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Baseline residual (actual - predicted)")
ax.set_ylabel("MW")
plt.show()

## Overall error

Already printed by `run_baseline()` above, repeated here for the record.

In [ ]:
print(f"MAPE: {utils.mape(actual, predicted):.2f}%")
print(f"RMSE: {utils.rmse(actual, predicted):.1f} MW")